# 3.4 Linear Regression Implementation from Scratch (PyTorch)

We are now ready to work through a fully functioning implementation of linear regression. In this section, we will implement the entire method from scratch, including:
1.  The model
2.  The loss function
3.  A minibatch stochastic gradient descent optimizer
4.  The training function that stitches all of these pieces together

Finally, we will run our synthetic data generator and apply our model on the resulting dataset.

In [5]:
%matplotlib inline
import torch
import random
from torch.utils import data
import matplotlib.pyplot as plt

## 3.4.1. Generating the Dataset
To simulate a real-world scenario, we generate a synthetic dataset where we know the true parameters. We construct a dataset according to the linear model:

$$\mathbf{y} = \mathbf{X} \mathbf{w} + b + \epsilon$$

Where $\epsilon$ is random noise.
* **True Weights ($\mathbf{w}$):** `[2, -3.4]`
* **True Bias ($b$):** `4.2`
* **Features:** 1000 examples with 2 features each.

In [6]:
def synthetic_data(w,b,num_examples):
    X=torch.normal(0,1,(num_examples,len(w)))
    y=torch.matmul(X,w)+b
    y+=torch.normal(0,0.01,y.shape)
    return X,y.reshape((-1,1))
true_w=torch.tensor([2,-3.4])
true_b=4.2
features,labels=synthetic_data(true_w,true_b,1000)
print('features:', features[0])
print('label:', labels[0])

features: tensor([ 0.4646, -2.4767])
label: tensor([13.5402])


## 3.4.2. Reading the Dataset
Training models requires iterating over the dataset and grabbing a small batch of data (a **minibatch**) at a time. This is essential for Stochastic Gradient Descent (SGD).

Instead of custom loaders, we use PyTorch's standard `TensorDataset` and `DataLoader`. These utilities handle shuffling and batching efficiently.

In [7]:
def load_array(data_arrays,batch_size,is_train=True):
    dataset=data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset,batch_size,shuffle=is_train)
batch_size=10
data_iter=load_array((features,labels),batch_size)
next(iter(data_iter))

[tensor([[ 0.7149,  0.9536],
         [ 1.1122, -0.5739],
         [-0.8454,  0.5382],
         [ 0.2120,  0.3968],
         [-0.7667, -1.3255],
         [-1.0123,  0.0126],
         [ 1.9425, -1.0561],
         [-1.8629,  1.4702],
         [-0.0793, -1.3320],
         [ 0.1939,  1.0806]]),
 tensor([[ 2.3828],
         [ 8.3554],
         [ 0.6783],
         [ 3.2656],
         [ 7.1737],
         [ 2.1438],
         [11.6538],
         [-4.5223],
         [ 8.5815],
         [ 0.9269]])]

## 3.4.3. Initializing Model Parameters
Before training, we must initialize the model parameters: weights $\mathbf{w}$ and bias $b$.
* **Weights ($\mathbf{w}$):** Initialized from a normal distribution with mean 0 and standard deviation 0.01.
* **Bias ($b$):** Initialized to 0.

Crucially, we set `requires_grad=True`. This tells PyTorch to track all operations on these tensors so it can automatically calculate gradients during backpropagation.

In [8]:
w=torch.normal(0,0.01,size=(2,1),requires_grad=True)
b=torch.zeros(1,requires_grad=True)

## 3.4.4. Defining the Model
We define the linear model. This relates the input to the output using matrix multiplication. 

The equation is:

$$\hat{\mathbf{y}} = \mathbf{X} \mathbf{w} + b$$

Note that $b$ is a scalar but is added to the vector result of $\mathbf{X} \mathbf{w}$ using Python's **broadcasting** mechanism.

In [9]:
def linreg(X,w,b):
    return torch.matmul(X,w)+b

## 3.4.5. Defining the Loss Function
We need a metric to measure how "wrong" our model is. We use the **squared loss** function:

$$L(\hat{\mathbf{y}}, \mathbf{y}) = \frac{1}{2} (\hat{\mathbf{y}} - \mathbf{y})^2$$

We divide by 2 to make the derivative cleaner (the 2 cancels out when differentiating).

In [10]:
def squared_loss(y_hat,y):
    return (y_hat-y.reshape(y_hat.shape))**2/2

## 3.4.6. Defining the Optimization Algorithm
We implement **Minibatch Stochastic Gradient Descent (SGD)**.
At each step, this function updates the parameters using their gradients. The size of the step is determined by the learning rate (`lr`).

We wrap this in `torch.no_grad()` because we are updating the weights in-place and don't want these update operations to be added to the computational graph (which is only for calculating gradients, not applying them).

In [11]:
def sgd(params,lr,batch_size):
    with torch.no_grad():
        for param in params:
            #param=param-lr*gradient
            param-=lr*param.grad/batch_size
            param.grad.zero_()


## 3.4.7. Training
Now we stitch all pieces together in the training loop. This is the core logic that most deep learning frameworks automate, but here we see it explicitly:

1.  **Iterate** through the data for a number of epochs.
2.  **Forward pass:** Compute predictions (`net(X, w, b)`) and loss (`loss(y_hat, y)`).
3.  **Backward pass:** Compute gradients (`l.sum().backward()`).
4.  **Update:** Adjust parameters using the optimizer (`sgd`).

In [12]:
lr=0.03
num_epochs=3
net=linreg
loss=squared_loss
for epoch in range(num_epochs):
    for X,y in data_iter:
        l=loss(net(X,w,b),y)
        #compute gradients
        l.sum().backward()
        
        #update param
        sgd([w,b],lr,batch_size)

    #not tracking gradient, only evaluating
    with torch.no_grad():
        train_l=loss(net(features,w,b),labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

epoch 1, loss 0.030184
epoch 2, loss 0.000120
epoch 3, loss 0.000052


In [13]:
print(f'error in estimating w: {true_w - w.reshape(true_w.shape)}')
print(f'error in estimating b: {true_b - b}')

error in estimating w: tensor([ 9.2065e-04, -9.1791e-05], grad_fn=<SubBackward0>)
error in estimating b: tensor([0.0008], grad_fn=<RsubBackward1>)
